In [ ]:
import os
import json
import textwrap
from typing import Any, Dict, List, Optional

from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.lib import colors
from reportlab.lib.units import mm
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Image as RLImage,
    PageBreak,
    Table,
    TableStyle,
)


def safe_get(d: Dict[str, Any], key: str, default=None):
    if not isinstance(d, dict):
        return default
    return d.get(key, default)


def audit_pipeline(intake: Dict[str, Any], research: Dict[str, Any], plan: Dict[str, Any]) -> Dict[str, Any]:
    issues: List[str] = []
    warnings: List[str] = []
    suggestions: List[str] = []

    destination = intake.get("destination")
    if destination and research.get("destination") and destination != research.get("destination"):
        issues.append("Destination mismatch between intake and research output.")

    if destination and plan.get("destination") and destination != plan.get("destination"):
        issues.append("Destination mismatch between intake and planner output.")

    days = intake.get("trip_duration_days")
    itinerary = plan.get("daily_itinerary", [])
    if isinstance(days, int) and isinstance(itinerary, list) and len(itinerary) != days:
        warnings.append(f"Planner returned {len(itinerary)} day objects, but intake requested {days} days.")

    # Count verification problems
    needs_verification_count = 0
    if isinstance(itinerary, list):
        for day in itinerary:
            if not isinstance(day, dict):
                continue
            for segment in ["morning", "afternoon", "evening"]:
                for item in day.get(segment, []) or []:
                    if isinstance(item, dict) and item.get("verification_status") == "needs_verification":
                        needs_verification_count += 1

    if needs_verification_count > 8:
        warnings.append(f"High number of unverified itinerary items: {needs_verification_count}.")

    if not plan.get("risk_notes"):
        suggestions.append("Planner output should include explicit risk notes.")

    if not plan.get("booking_notes"):
        suggestions.append("Planner output should include booking notes.")

    if not research.get("constraints"):
        suggestions.append("Research output should carry constraints for the planner.")

    return {
        "issues": issues,
        "warnings": warnings,
        "suggestions": suggestions,
    }


# -------------------------------------------------
# 2) Enrichment stubs
#    Replace these with real data sources later
# -------------------------------------------------
def get_map_image(destination: str, out_path: str) -> Optional[str]:
    """
    Use Google Maps Static API or a browser screenshot here.
    Return a local image path if successful.
    """
    # Placeholder: if you already have a screenshot file, copy it here.
    return None


def get_weather_advisory(intake: Dict[str, Any]) -> str:
    travel_date = intake.get("travel_date") or ""
    destination = intake.get("destination", "your destination")
    if not travel_date:
        return f"Travel date is unspecified, so use seasonal weather guidance for {destination} and check forecast again before departure."
    return f"Check the forecast for {destination} around {travel_date}. If conditions are unstable, keep one indoor backup block each day."


def get_safety_advisory(destination: str) -> List[str]:
    return [
        "Keep valuables split across two places, not all in one bag.",
        "Use official transport or reputable ride-hailing at night.",
        "Save hotel address in local language if possible.",
    ]


def get_cultural_advisory(destination: str) -> List[str]:
    return [
        "Check dress expectations for temples, religious sites, or formal restaurants.",
        "Learn a few local courtesy phrases before arrival.",
        "Keep volume low in shared transport and quiet public spaces.",
    ]


# -------------------------------------------------
# 3) Build final report JSON
# -------------------------------------------------
def build_final_report(
    intake: Dict[str, Any],
    research: Dict[str, Any],
    plan: Dict[str, Any],
    writer_md: str,
    map_image_path: Optional[str] = None,
) -> Dict[str, Any]:
    audit = audit_pipeline(intake, research, plan)

    report = {
        "meta": {
            "schema_version": "1.0",
            "destination": intake.get("destination"),
            "trip_duration_days": intake.get("trip_duration_days"),
        },
        "audit": audit,
        "enrichment": {
            "weather_advisory": get_weather_advisory(intake),
            "safety_advisory": get_safety_advisory(intake.get("destination", "")),
            "cultural_advisory": get_cultural_advisory(intake.get("destination", "")),
            "map_image_path": map_image_path,
        },
        "source_data": {
            "intake": intake,
            "research": research,
            "plan": plan,
        },
        "writer_markdown": writer_md,
    }
    return report


# -------------------------------------------------
# 4) Render PDF
# -------------------------------------------------
def _add_wrapped_paragraph(story, styles, title: str, text: str):
    if title:
        story.append(Paragraph(title, styles["Heading2"]))
    for para in text.split("\n"):
        para = para.strip()
        if para:
            story.append(Paragraph(para.replace("&", "&amp;"), styles["BodyText"]))
            story.append(Spacer(1, 4))


def render_pdf(report: Dict[str, Any], output_path: str) -> str:
    styles = getSampleStyleSheet()
    styles.add(
        ParagraphStyle(
            name="CenterTitle",
            parent=styles["Title"],
            alignment=TA_CENTER,
            textColor=colors.HexColor("#111111"),
        )
    )

    doc = SimpleDocTemplate(
        output_path,
        pagesize=A4,
        rightMargin=18 * mm,
        leftMargin=18 * mm,
        topMargin=16 * mm,
        bottomMargin=16 * mm,
    )

    story = []

    meta = report.get("meta", {})
    audit = report.get("audit", {})
    enrichment = report.get("enrichment", {})
    writer_md = report.get("writer_markdown", "")

    story.append(Paragraph("Final Travel Report", styles["CenterTitle"]))
    story.append(Spacer(1, 10))

    story.append(Paragraph(f"Destination: {meta.get('destination', 'Unknown')}", styles["Heading2"]))
    story.append(Paragraph(f"Trip Duration: {meta.get('trip_duration_days', 'Unknown')} days", styles["BodyText"]))
    story.append(Spacer(1, 8))

    # Map image
    map_path = enrichment.get("map_image_path")
    if map_path and os.path.exists(map_path):
        story.append(Paragraph("Destination Map", styles["Heading2"]))
        story.append(RLImage(map_path, width=170 * mm, height=95 * mm))
        story.append(Spacer(1, 8))

    # Audit section
    story.append(Paragraph("Audit Summary", styles["Heading2"]))
    if audit.get("issues"):
        for x in audit["issues"]:
            story.append(Paragraph(f"• {x}", styles["BodyText"]))
    else:
        story.append(Paragraph("No hard inconsistencies found.", styles["BodyText"]))

    if audit.get("warnings"):
        story.append(Spacer(1, 4))
        story.append(Paragraph("Warnings", styles["Heading3"]))
        for x in audit["warnings"]:
            story.append(Paragraph(f"• {x}", styles["BodyText"]))

    if audit.get("suggestions"):
        story.append(Spacer(1, 4))
        story.append(Paragraph("Suggestions", styles["Heading3"]))
        for x in audit["suggestions"]:
            story.append(Paragraph(f"• {x}", styles["BodyText"]))

    story.append(PageBreak())

    # Advisories
    story.append(Paragraph("Weather / Safety / Culture", styles["Heading2"]))
    story.append(Paragraph(enrichment.get("weather_advisory", ""), styles["BodyText"]))
    story.append(Spacer(1, 6))

    story.append(Paragraph("Safety Notes", styles["Heading3"]))
    for x in enrichment.get("safety_advisory", []):
        story.append(Paragraph(f"• {x}", styles["BodyText"]))

    story.append(Spacer(1, 6))
    story.append(Paragraph("Cultural Notes", styles["Heading3"]))
    for x in enrichment.get("cultural_advisory", []):
        story.append(Paragraph(f"• {x}", styles["BodyText"]))

    story.append(PageBreak())

    # Writer markdown as appendix text
    story.append(Paragraph("Travel Guide (Writer Output)", styles["Heading2"]))
    for block in writer_md.split("\n"):
        block = block.strip()
        if block.startswith("# "):
            story.append(Paragraph(block[2:], styles["Heading1"]))
        elif block.startswith("## "):
            story.append(Paragraph(block[3:], styles["Heading2"]))
        elif block.startswith("### "):
            story.append(Paragraph(block[4:], styles["Heading3"]))
        elif block:
            safe = block.replace("&", "&amp;")
            story.append(Paragraph(safe, styles["BodyText"]))
        else:
            story.append(Spacer(1, 4))

    doc.build(story)
    return output_path


# -------------------------------------------------
# 5) Main entry
# -------------------------------------------------
def main():
    base_dir = "/content/drive/MyDrive/547 Final Project/agents"

    with open(os.path.join(base_dir, "initial_output.json"), "r", encoding="utf-8") as f:
        intake = json.load(f)

    with open(os.path.join(base_dir, "research_output.json"), "r", encoding="utf-8") as f:
        research = json.load(f)

    with open(os.path.join(base_dir, "planner_output.json"), "r", encoding="utf-8") as f:
        plan = json.load(f)

    with open(os.path.join(base_dir, "writer_output.md"), "r", encoding="utf-8") as f:
        writer_md = f.read()

    map_path = get_map_image(intake.get("destination", ""), os.path.join(base_dir, "map.png"))
    report = build_final_report(intake, research, plan, writer_md, map_image_path=map_path)

    with open(os.path.join(base_dir, "final_report.json"), "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    pdf_path = os.path.join(base_dir, "final_report.pdf")
    render_pdf(report, pdf_path)
    print(f"Saved PDF to: {pdf_path}")


if __name__ == "__main__":
    main()